# 14 — Comparaison globale mise à jour

Ce notebook ne réentraîne aucun modèle. Il charge les artefacts sauvegardés. Le « gagnant du protocole » est le modèle choisi selon le macro-F1 de validation. La « généralisation test observée » décrit seulement la performance mesurée après gel du modèle, de la perte et du seuil ; elle ne sert pas à réviser la sélection.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import *
from src.helpers import count_parameters, set_seed

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Périphérique sélectionné :", device)
print("CUDA disponible :", torch.cuda.is_available())

In [ ]:
MODEL_DIRS = {
    "MLP statistique": "mlp",
    "CNN 2D": "cnn2d",
    "RNN": "rnn_fair" if (PROJECT_ROOT / "artifacts/models/rnn_fair").exists() else "rnn",
    "LSTM": "lstm_fair" if (PROJECT_ROOT / "artifacts/models/lstm_fair").exists() else "lstm",
}

def load_model_row(display_name, directory):
    path = PROJECT_ROOT / "artifacts/models" / directory
    required = ["model_config.json", "validation_metrics.json", "test_metrics.json", "training_summary.json"]
    if not all((path / name).exists() for name in required): return None
    values = []
    for name in required:
        with open(path / name, encoding="utf-8") as handle: values.append(json.load(handle))
    config, validation, test, summary = values
    return {"model": display_name, "parameter count": config.get("parameter_count"),
        "validation macro F1": validation.get("macro_f1"), "test macro F1": test.get("macro_f1"),
        "weighted F1": test.get("weighted_f1"), "stress precision": test.get("stress_precision"),
        "stress recall": test.get("stress_recall"), "ROC-AUC": test.get("roc_auc"),
        "average precision": test.get("average_precision"), "best epoch": summary.get("best_epoch"),
        "training time": summary.get("training_time_seconds"), "inference time": test.get("inference_time_seconds"),
        "artifact directory": directory}

rows = [row for name, directory in MODEL_DIRS.items() if (row := load_model_row(name, directory)) is not None]
comparison = pd.DataFrame(rows)
display(comparison)

## Gagnant du protocole et généralisation test observée

In [ ]:
if len(comparison):
    protocol_winner = comparison.loc[comparison["validation macro F1"].idxmax()]
    print("Gagnant du protocole (validation) :", protocol_winner["model"])
    print("Sa généralisation test observée :", protocol_winner["test macro F1"])
    axes = comparison.set_index("model")[["validation macro F1", "test macro F1"]].plot.bar(figsize=(9, 4), rot=20)
    axes.set_ylabel("Macro-F1"); axes.set_title("Validation contre test (test non utilisé pour sélectionner)"); plt.tight_layout(); plt.show()

## Coût et métriques de la classe stress

In [ ]:
if len(comparison):
    display(comparison[["model", "parameter count"]].sort_values("parameter count"))
    display(comparison[["model", "training time", "inference time"]])
    display(comparison[["model", "stress precision", "stress recall", "ROC-AUC", "average precision"]])

## Résultats par sujet

In [ ]:
subject_frames = []
for _, row in comparison.iterrows():
    path = PROJECT_ROOT / "artifacts/models" / row["artifact directory"] / "per_subject_metrics.csv"
    if path.exists():
        frame = pd.read_csv(path); frame.insert(0, "model", row["model"]); subject_frames.append(frame)
per_subject_all = pd.concat(subject_frames, ignore_index=True) if subject_frames else pd.DataFrame()
display(per_subject_all)

## Synthèse conceptuelle

- **MLP** : représentation statistique globale ;
- **CNN 2D** : motifs temps-fréquence locaux dans les scalogrammes ;
- **RNN/LSTM** : dépendances temporelles ordonnées.

Une complexité ou un nombre de paramètres supérieur ne garantit pas une meilleure généralisation indépendante des sujets. WESAD ne fournit ici que quinze participants indépendants : les écarts entre validation et test peuvent refléter une forte sensibilité aux personnes composant chaque sous-ensemble. L'interprétation doit donc associer macro-F1, rappel stress et résultats par sujet, tout en conservant la distinction stricte entre choix sur validation et observation finale sur test.